# Day 10 — Mock Interview Practice
**Week 2 · Day 3 · Oct 22, 2026** — Difficulty: **INTERMEDIATE → ADVANCED** (7 INTERMEDIATE, 11 ADVANCED)

18 questions across 8 categories. Attempt every question on paper or a scratch cell **before** opening the solution. Time yourself: INTERMEDIATE ≈ 15 min, ADVANCED ≈ 25 min.

| # | Section | Q | Topics |
|---|---|---|---|
| 1 | SQL Advanced | Q1–Q2 | conversion funnel with window functions · exact vs approximate distinct counting at scale |
| 2 | Python — Loops/Functions/Structures | Q3–Q4 | merge overlapping intervals · running median (two heaps) |
| 3 | PySpark Coding | Q5–Q6 | corrupt-record quarantine · group-wise scoring with applyInPandas |
| 4 | Spark Optimization | Q7–Q8 | shuffle elimination (bucketing vs liquid clustering) · Tungsten & whole-stage codegen |
| 5 | ETL — Failures, Late Data, Schema | Q9–Q11 | file-ingestion control-table ledger · transactional outbox · _rescued_data drift backfill |
| 6 | Data Modeling | Q12–Q13 | effective-dated currency conversion · mini-dimension for fast-changing attributes |
| 7 | Snowflake | Q14–Q15 | table types (transient/permanent/external) · resource monitors & multi-cluster warehouses |
| 8 | Monitoring, Logging & Orchestration | Q16–Q18 | Jobs for-each tasks & matrix · driver logs & cluster log delivery · Airflow TaskFlow + deferrable operators |

**Score card at the bottom.** Fill it honestly — it drives what you review tomorrow.

## Q1 — E-commerce Conversion Funnel with Window Functions `[INTERMEDIATE]`

You are given an `events` table with `user_id`, `event_ts`, `event_type` (`view`, `cart`, `purchase`). Product wants today's **conversion funnel**:

* Step 1: users who viewed, Step 2: users who viewed **then** carted, Step 3: users who viewed **then** carted **then** purchased.
* A user only counts for a step if they reached **every earlier step**, in that **timestamp order** (cart before view does NOT count; purchase without cart does NOT count).
* Output: users at each step + step-to-step conversion rates.

**Follow-up the interviewer will ask:** how would you generalize this to N arbitrary steps, and what's wrong with the naive `COUNT(DISTINCT CASE WHEN ...)` approach?

In [0]:
# Solution Q1 — funnel via conditional MIN + ordered progression check
from pyspark.sql import functions as F

events = spark.createDataFrame([
    ("u1", "2026-10-20 10:00:00", "view"), ("u1", "2026-10-20 10:05:00", "cart"),
    ("u1", "2026-10-20 10:10:00", "purchase"), ("u2", "2026-10-20 11:00:00", "view"),
    ("u2", "2026-10-20 11:20:00", "purchase"),           # purchase WITHOUT cart -> must NOT count
    ("u3", "2026-10-20 12:00:00", "view"),               # view only
    ("u4", "2026-10-20 12:30:00", "view"), ("u4", "2026-10-20 12:40:00", "cart"),
    ("u5", "2026-10-20 13:00:00", "cart"),               # cart WITHOUT view -> must NOT count
], ["user_id", "event_ts", "event_type"])
events.createOrReplaceTempView("events")

funnel = spark.sql("""
WITH step_times AS (                       -- 1) first timestamp of each step per user
  SELECT user_id,
         MIN(CASE WHEN event_type = 'view'     THEN event_ts END) AS t_view,
         MIN(CASE WHEN event_type = 'cart'     THEN event_ts END) AS t_cart,
         MIN(CASE WHEN event_type = 'purchase' THEN event_ts END) AS t_purchase
  FROM events
  GROUP BY user_id
),
progressed AS (                            -- 2) keep a step only if reached IN ORDER
  SELECT user_id, t_view,
         CASE WHEN t_cart     >= t_view THEN t_cart     END AS t_cart,
         CASE WHEN t_purchase >= t_view
               AND (t_cart IS NULL OR t_purchase >= t_cart)
              THEN t_purchase END AS t_purchase
  FROM step_times
)
SELECT SUM(CASE WHEN t_view     IS NOT NULL THEN 1 ELSE 0 END) AS step1_view,
       SUM(CASE WHEN t_cart      IS NOT NULL THEN 1 ELSE 0 END) AS step2_cart,
       SUM(CASE WHEN t_purchase  IS NOT NULL THEN 1 ELSE 0 END) AS step3_purchase,
       ROUND(SUM(CASE WHEN t_cart     IS NOT NULL THEN 1 ELSE 0 END)
             / SUM(CASE WHEN t_view IS NOT NULL THEN 1 ELSE 0 END) * 100, 1) AS view_to_cart_pct,
       ROUND(SUM(CASE WHEN t_purchase IS NOT NULL THEN 1 ELSE 0 END)
             / SUM(CASE WHEN t_cart  IS NOT NULL THEN 1 ELSE 0 END) * 100, 1) AS cart_to_purchase_pct
FROM progressed
""")
funnel.show()

r = funnel.collect()[0]
assert (r["step1_view"], r["step2_cart"], r["step3_purchase"]) == (4, 2, 2), r  # u5 excluded, u2 excluded from step3
print("funnel assertions passed")

# INTERVIEW TALKING POINTS
# - Naive COUNT(DISTINCT CASE WHEN event='cart') OVER-counts: it ignores step ORDER. The two-CTE
#   pattern (conditional MIN -> ordered-keep) fixes that and stays a single shuffle (GROUP BY user).
# - Generalizing to N steps: build a steps config table (step_name, step_order), join events to it,
#   conditional MIN per step, then "keep step n if t_n >= t_(n-1)" with LAST_VALUE IGNORE NULLS
#   over step_order -- same idea, data-driven.
# - Funnel vs cohort (Day 8): funnel = ordered steps within a session/day; cohort = retention across
#   weeks. Interviewers love when you proactively distinguish them.
# - In a real pipeline, sessionize first (30-min inactivity windows, Day 8 gaps-and-islands) and
#   compute the funnel per session, not per user-day.

## Q2 — Exact vs Approximate Distinct Counting at Scale `[ADVANCED]`

Your events table gets **2 billion rows/day**. The DAU dashboard runs `COUNT(DISTINCT user_id)` grouped by day and takes minutes, shuffling terabytes. Explain and demonstrate:

1. Why `COUNT(DISTINCT)` is expensive (what does it force on the shuffle? what are the memory constraints with hundreds of millions of distinct keys?).
2. How `APPROX_COUNT_DISTINCT` / HyperLogLog works internally (sketches, registers, error bound) and what ~2% error means in practice.
3. When approximate is acceptable vs when you must stay exact (dashboard vs billing vs A/B significance).
4. How the same trade-off appears in PySpark (`approx_count_distinct` with `rsd`), and one strategy to keep exact counting cheap anyway.

In [0]:
# Solution Q2 — COUNT(DISTINCT) vs APPROX_COUNT_DISTINCT, measured
from pyspark.sql import functions as F
import time

big = spark.range(2_000_000).withColumn("grp", (F.col("id") % 20).cast("string"))
big.createOrReplaceTempView("big")

# 1) EXACT — one shuffle of ALL rows, per-group hash-set aggregation
spark.sql("SELECT COUNT(DISTINCT id) c FROM big").collect()          # warm up

t0 = time.time()
exact = spark.sql("SELECT grp, COUNT(DISTINCT id) exact_cnt FROM big GROUP BY grp").collect()
t_exact = time.time() - t0

# 2) APPROXIMATE — per-partition HLL sketch; shuffle carries tiny sketch bytes, not raw rows
t0 = time.time()
approx = spark.sql("SELECT grp, APPROX_COUNT_DISTINCT(id) approx_cnt FROM big GROUP BY grp").collect()
t_approx = time.time() - t0

ex = {r["grp"]: r["exact_cnt"] for r in exact}
ap = {r["grp"]: r["approx_cnt"] for r in approx}
print(f"exact   : {t_exact:.2f}s")
print(f"approx  : {t_approx:.2f}s")
errs = {g: abs(ap[g] - ex[g]) / ex[g] * 100 for g in ex}
avg_err = sum(errs.values()) / len(errs)
worst_g, max_err = max(errs.items(), key=lambda kv: kv[1])
print(f"avg relative error: {avg_err:.2f}% | worst group {worst_g}: {max_err:.2f}%")
print("(HLL is statistical: per-group error varies run to run — that's exactly the trade-off)")
assert avg_err < 5.0, "default rsd 0.05 -> average error should stay a few percent"

# PySpark API with tunable relative standard deviation (default rsd=0.05 -> ~4096 registers)
df = big.groupBy("grp").agg(
    F.approx_count_distinct("id").alias("approx_default"),
    F.approx_count_distinct("id", rsd=0.01).alias("approx_rsd_1pct"),   # more registers = tighter error, still tiny
)
df.orderBy("grp").show(3)

# INTERVIEW TALKING POINTS
# - COUNT(DISTINCT) forces a shuffle keyed by (group, id) carrying EVERY row, then builds per-group
#   hash sets; with 300M distinct users the sets spill. APPROX hashes ids into m registers tracking
#   max leading-zero count; Spark merges sketches (element-wise max) and estimates with the harmonic
#   mean -> error ~1.04/sqrt(m). rsd=0.05 -> m~4096; rsd=0.01 -> m~16384.
# - 2% error on DAU 180M = ±3.6M. Fine for trend dashboards; NOT fine for billing, refunds, or
#   A/B significance. Decision rule to state out loud: "approx for product analytics, exact where
#   money changes hands."
# - Keeping exact cheap: (a) two-phase exact -- per-partition GROUP BY id, then COUNT the reduced
#   set (shuffles distinct ids once, not every row); (b) pre-materialized daily distincts;
#   (c) ask if the business question even needs cross-day dedup.
# - Name-drop family: Theta sketches (set intersection/union), KLL (quantiles), Bloom filters
#   (membership) -- the sketch mindset is what interviewers test with this question.

## Q3 — Merge Overlapping Intervals `[INTERMEDIATE]`

Given a list of meeting time intervals `[[start, end], ...]`, merge all overlapping intervals and return the merged list sorted by start. A classic that shows up at Google, Amazon and every phone screen.

* Input: `[[1,3],[2,6],[8,10],[15,18]]` → `[[1,6],[8,10],[15,18]]` (first two overlap; touching intervals like `[1,4],[4,5]` also merge).
* **Follow-up 1:** what's the complexity, and where does the sort cost dominate?
* **Follow-up 2 (LeetCode 57):** given a *sorted, disjoint* list and ONE new interval, insert and merge it in **O(n)** without re-sorting.

In [0]:
# Solution Q3 — sort + linear sweep
def merge_intervals(intervals):
    """O(n log n) time, O(n) space. Sort by start, sweep with a running anchor."""
    if not intervals:
        return []
    ivs = sorted(intervals, key=lambda x: x[0])          # sort dominates: O(n log n)
    merged = [list(ivs[0])]
    for start, end in ivs[1:]:
        last = merged[-1]
        if start <= last[1]:                               # overlaps OR touches last[1]
            last[1] = max(last[1], end)                   # extend (max handles engulfed intervals)
        else:
            merged.append([start, end])                  # gap -> new interval
    return merged

# tests
assert merge_intervals([[1,3],[2,6],[8,10],[15,18]]) == [[1,6],[8,10],[15,18]]
assert merge_intervals([[1,4],[4,5]]) == [[1,5]]          # touching counts as overlap
assert merge_intervals([[1,10],[2,3],[3,4]]) == [[1,10]]   # fully engulfed
assert merge_intervals([]) == []
assert merge_intervals([[5,7]]) == [[5,7]]
assert merge_intervals([[2,2],[2,2]]) == [[2,2]]           # zero-length
print("merge_intervals: all tests passed")

# Follow-up 2: insert ONE interval into a SORTED DISJOINT list in O(n)
def insert_interval(intervals, new):
    out, i, n = [], 0, len(intervals)
    while i < n and intervals[i][1] < new[0]:              # 1) copy everything ending before new starts
        out.append(intervals[i]); i += 1
    while i < n and intervals[i][0] <= new[1]:             # 2) absorb everything overlapping new
        new = [min(new[0], intervals[i][0]), max(new[1], intervals[i][1])]
        i += 1
    out.append(list(new))                                  # 3) the merged new interval
    while i < n:                                           # 4) copy the rest
        out.append(intervals[i]); i += 1
    return out

assert insert_interval([[1,3],[6,9]], [2,5]) == [[1,5],[6,9]]
assert insert_interval([[1,2],[3,5],[6,7],[8,10],[12,16]], [4,8]) == [[1,2],[3,10],[12,16]]
print("insert_interval: all tests passed")

# INTERVIEW TALKING POINTS
# - The sweep invariant: keep ONE anchor interval; a new interval either overlaps (extend end with
#   max) or starts a gap (flush anchor, start new). Sorting by start guarantees every later interval
#   either overlaps the anchor or nothing before it.
# - Variants to mention: calendar-free-busy (merge per person first, then intersect two merged
#   lists with a two-pointer scan), employee-free-time, minimum platforms (sort starts & ends
#   separately, walk both).
# - PySpark angle if asked: intervals -> sessionization with window LAG(end) OVER (ORDER BY start),
#   cumulative max of end, then flag gaps = exactly the gaps-and-islands pattern from Day 8.

## Q4 — Running Median from a Data Stream `[ADVANCED]`

Design a class supporting two operations on an infinite stream of numbers:

* `add_num(x)` — append a number, **O(log n)**
* `find_median()` — return the current median, **O(1)**

`(LeetCode 295 — asked at Google, Amazon, Uber; Uber's variant is the *sliding-window* median over the last k readings.)

Explain the data structure invariant precisely, why it beats a sorted list, and how you'd extend it to the sliding-window variant where elements also leave the window.

In [0]:
# Solution Q4 — two heaps: max-heap of the lower half, min-heap of the upper half
import heapq

class MedianFinder:
    def __init__(self):
        self.lo = []   # lower half as NEGATED values -> heapq (min-heap) behaves as a max-heap
        self.hi = []   # upper half, plain min-heap

    def add_num(self, x):
        # invariant: len(lo) - len(hi) in {0, 1}; every element of lo <= every element of hi
        if not self.lo or x <= -self.lo[0]:
            heapq.heappush(self.lo, -x)
        else:
            heapq.heappush(self.hi, x)
        # rebalance sizes
        if len(self.lo) - len(self.hi) > 1:                      # lo too big by 2
            heapq.heappush(self.hi, -heapq.heappop(self.lo))
        elif len(self.hi) - len(self.lo) > 0:                    # hi bigger (not allowed)
            heapq.heappush(self.lo, -heapq.heappop(self.hi))
        return self

    def find_median(self):
        if len(self.lo) > len(self.hi):                         # odd count -> lo holds the middle
            return float(-self.lo[0])
        return (-self.lo[0] + self.hi[0]) / 2.0                 # even -> average of the two tops

mf = MedianFinder()
for x, expected in [(2, 2), (3, 2.5), (4, 3), (5, 3.5), (1, 3), (6, 3.5), (0, 3)]:
    mf.add_num(x)
    assert mf.find_median() == expected, (x, mf.find_median(), expected)
print("MedianFinder: all tests passed")

# sliding-window variant (Uber-style): median over the last k readings (LeetCode 480)
class WindowMedian:
    """Two heaps + explicit removal. list.remove + heapify is O(k) per eviction — fine for
    window sizes; at scale use LAZY DELETION (a Counter of pending departures, pruned only
    when a ghost reaches a heap top) to keep the O(log k) bound. Say both in the interview."""
    def __init__(self):
        self.small = []   # max-heap of the lower half (negated values)
        self.large = []   # min-heap of the upper half

    def _rebalance(self):
        if len(self.small) > len(self.large) + 1:
            heapq.heappush(self.large, -heapq.heappop(self.small))
        elif len(self.large) > len(self.small):
            heapq.heappush(self.small, -heapq.heappop(self.large))

    def add(self, x):
        if not self.small or x <= -self.small[0]:
            heapq.heappush(self.small, -x)
        else:
            heapq.heappush(self.large, x)
        self._rebalance()

    def remove(self, x):
        try:
            self.small.remove(-x); heapq.heapify(self.small)   # O(k) removal
        except ValueError:
            self.large.remove(x);  heapq.heapify(self.large)
        self._rebalance()

    def median(self):
        n = len(self.small) + len(self.large)
        if n == 0: return None
        if n % 2 == 1:
            return float(-self.small[0]) if len(self.small) >= len(self.large) else float(self.large[0])
        return (-self.small[0] + self.large[0]) / 2

def sliding_window_median(nums, k):
    wm, out = WindowMedian(), []
    for i, x in enumerate(nums):
        wm.add(x)
        if i >= k: wm.remove(nums[i - k])
        if i >= k - 1: out.append(wm.median())
    return out

assert sliding_window_median([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1, -1, -1, 3, 5, 6]
print("WindowMedian: all tests passed")

# INTERVIEW TALKING POINTS
# - Invariant is THE answer: lower half in a max-heap, upper in a min-heap, sizes balanced within 1.
#   Median = top of the bigger heap (odd) or average of both tops (even). Python has no max-heap,
#   so negate values — mention that explicitly, it's a common fumble.
# - vs sorted list: add is O(n) (shift) or O(log n) with bisect + O(n) insert; two heaps give O(log n)
#   add and O(1) median with only two integer comparisons per query.
# - Sliding window: you cannot delete arbitrary heap elements — lazy deletion + a Counter of pending
#   departures, pruned when ghosts reach the top. Alternative: TWO multisets / an indexable skiplist
#   (Order Statistics Tree) for O(log n) delete + order queries.
# - Data-eng parallel: exact percentiles in Spark = approx_percentile/percentile_approx, which is a
#   GK/sketch-based single-pass approximation — same "streaming + bounded memory" mindset.

## Q5 — Quarantining Corrupt Records Without Failing the Job `[INTERMEDIATE]`

A partner drops JSON clickstream files into your landing zone; ~0.1% of lines are malformed (truncated writes during upload, encoding glitches). Requirements:

1. The ingestion job must **not fail** on bad lines.
2. Good rows land in a bronze table; bad rows land in a **quarantine table** with the raw payload **and the source file path**, so the partner can be sent a fix-list.
3. A data-quality metric (bad-row ratio) must be logged per run.

Show the Spark mechanism you'd use, and explain `PERMISSIVE` vs `FAILFAST` vs `DROPMALFORMED`.

In [0]:
# Solution Q5 — PERMISSIVE mode + columnNameOfCorruptRecord, split good/bad, quarantine with file path
import os
from pyspark.sql import functions as F

BASE = "/Volumes/workspace/default/temp_parquet/day10"
os.makedirs(BASE + "/raw_bad", exist_ok=True)
with open(BASE + "/raw_bad/f1.json", "w") as f:                 # simulate the partner drop
    f.write('{"id": 1, "name": "ann"}\n'
            '{"id": 2, "name": "bob"}\n'
            '{this line got truncated\n'                        # malformed
            '{"id": 3, "name": "cat"}\n')

# PERMISSIVE: bad rows parse with NULLs; the raw text goes into a corrupt-record column
parsed = (spark.read
          .option("mode", "PERMISSIVE")
          .option("columnNameOfCorruptRecord", "_corrupt_record")
          .json(BASE + "/raw_bad"))
# Unity Catalog blocks queries that touch the corrupt-record column directly on raw JSON —
# it forces MATERIALIZATION first, which is also the right production pattern: land the raw
# parse once (bronze-raw), then split good/bad from the materialized copy.
(parsed.write.mode("overwrite").parquet(BASE + "/parsed_stage"))
staged = spark.read.parquet(BASE + "/parsed_stage")
staged.createOrReplaceTempView("parsed")

bronze = spark.sql("SELECT id, name FROM parsed WHERE _corrupt_record IS NULL")
quarantine = spark.sql("""SELECT id, _corrupt_record AS raw_payload
                          FROM parsed WHERE _corrupt_record IS NOT NULL""")
bronze.show(); quarantine.show(truncate=False)
assert bronze.count() == 3 and quarantine.count() == 1

# In production, enrich quarantine with F.col("_metadata.file_path") — the source file —
# (Unity Catalog forbids the legacy input_file_name()) and write both to Delta, then the run metric:
bad_ratio = quarantine.count() / (bronze.count() + quarantine.count())
print(f"bad-row ratio this run: {bad_ratio:.2%}")

# INTERVIEW TALKING POINTS
# - Parser modes: PERMISSIVE (nulls + corrupt column, default), DROPMALFORMED (silently drop —
#   dangerous: you lose evidence and volume metrics), FAILFAST (abort — right when the contract is
#   strict, e.g. financial feeds). Quarantine-with-evidence > silent drop, always.
# - _corrupt_record is only reliably populated for row-oriented formats (JSON/CSV). For parquet a
#   bad file fails at read time — handle with a file-level try/except or an Auto Loader quarantine
#   directory.
# - Anti-pattern to call out: catch-all except + skip-file in a loop hides systematic breakage;
#   pair the bad-ratio metric with an alert (>1% for 3 consecutive runs = page a human). This is
#   your Day 8 DQ-gate pattern in disguise: measure, quarantine, alert.
# - If the SAME schema drift (not truncation) causes nulls, that's a DIFFERENT mechanism:
#   _rescued_data (Q11) — say the distinction out loud, interviewers test whether you confuse them.

## Q6 — Group-Wise Scoring with `applyInPandas` `[ADVANCED]`

You must flag anomalous daily sales for **each store independently** — each store's amounts get a z-score computed **within that store's own history** (a $900 day is wild for a kiosk, normal for a flagship). Millions of rows, thousands of stores.

1. Implement it with `groupBy(...).applyInPandas(...)` (a *grouped-map* pandas function).
2. Explain exactly what crosses the JVM→Python boundary (Arrow batches), why this beats a classic row-at-a-time UDF, and why a grouped-map pandas UDF requires an explicit output schema.
3. Compare three implementation options and when each wins: native window functions vs `applyInPandas` vs plain UDF.

In [0]:
# Solution Q6 — grouped z-scores with applyInPandas
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType
import pandas as pd

sales = spark.createDataFrame(
    [(1,"a",10.0),(2,"a",20.0),(3,"a",30.0),(4,"b",5.0),(5,"b",7.0),(6,"b",9.0)],
    ["row_id", "store", "amt"])

schema = StructType([StructField("row_id", LongType()),
                    StructField("store", StringType()),
                    StructField("amt", DoubleType()),
                    StructField("amt_z", DoubleType())])

def zscore(pdf: pd.DataFrame) -> pd.DataFrame:
    """Receives ALL rows of ONE store as a single pandas DataFrame (Arrow batch)."""
    m, s = pdf["amt"].mean(), pdf["amt"].std(ddof=0)
    pdf["amt_z"] = (pdf["amt"] - m) / s if s > 0 else 0.0
    return pdf

out = (sales.groupBy("store")
            .applyInPandas(zscore, schema=schema)     # grouped-map pandas function
            .orderBy("row_id"))
out.show()
assert out.filter("row_id = 1").collect()[0]["amt_z"] < -1                # first point is below mean
assert abs(out.filter("row_id = 2").collect()[0]["amt_z"]) < 1e-9         # middle point IS the mean
print("applyInPandas assertions passed")

# INTERVIEW TALKING POINTS
# - Mechanics: Spark shuffles by group key, then hands EACH group to Python as one Arrow
#   (columnar, zero-serialization-overhead) batch -> your function returns a full DataFrame.
#   Output schema is mandatory because Spark plans the query BEFORE Python runs — it cannot
#   infer types it hasn't seen.
# - vs classic UDF: row-at-a-time Python UDFs pay per-row serialization + GIL contention and are
#   black boxes to the optimizer (no pushdown/predicate movement INTO the udf). Arrow-batched
#   pandas UDFs amortize serialization over thousands of rows and use vectorized pandas/numpy ops.
# - vs native window: this exact z-score CAN be done with window functions
#   ((amt - avg(amt) OVER (PARTITION BY store)) / stddev(amt) OVER (PARTITION BY store)) and that
#   version is FASTER (no Python). applyInPandas wins when the per-group logic doesn't map to
#   expressions: per-group ML model scoring, per-group quantile bucketing, group-wise text parsing.
#   Rule of thumb: "expressible in Spark SQL -> window; needs a library per group -> applyInPandas.""
# - Sibling APIs: mapInPandas (flat map, no grouping), cogroup.applyInPandas (two DataFrames
#   grouped by the same key — e.g. join two series per store).
# - Gotchas: rows returned for OTHER groups silently disappear; an empty return DataFrame drops
#   the group; pandas dtypes must match the declared schema (int64 vs int32 mismatch throws).

## Q7 — Eliminating the Shuffle in a Repeated Join: Bucketing `[INTERMEDIATE]`

A reporting job joins `orders` (500 GB) to `customers` (200 GB) every hour. Every run pays a full shuffle of **both** sides. Explain:

1. How **bucketing** removes that exchange: pre-shuffle both tables once into the same number of buckets on the join key; why tasks can then join bucket-to-bucket locally (sort-merge / shuffle-hash without Exchange).
2. How to choose the bucket count (and the skew trap that turns one bucket into a straggler).
3. On Databricks + Unity Catalog, managed tables are Delta and Hive bucketing is not the tool — what replaces it (**liquid clustering `CLUSTER BY`**, ZORDER/OPTIMIZE lineage) and what the plan looks like when pruning works.

Demonstrate the Databricks-native part; show the classic bucketing DDL as the talking point.

In [0]:
# Solution Q7 — liquid clustering (Databricks UC) + classic bucketing DDL as talking points
from pyspark.sql import functions as F

items = spark.createDataFrame([(i, f"item_{i}", i % 50) for i in range(1000)],
                              ["item_id", "item_name", "category_id"])
items.createOrReplaceTempView("items")

# --- Databricks UC (Delta): liquid clustering instead of Hive buckets -------------------
# Data is CLUSTERED BY the key; as data is written, rows land in files already sorted by the
# cluster key -> files carry min/max key metadata -> joins and filters PRUNE whole files.
spark.sql("DROP TABLE IF EXISTS workspace.default.day10_lc_items")
spark.sql("""CREATE TABLE workspace.default.day10_lc_items
             (item_id BIGINT, item_name STRING, category_id BIGINT)
             CLUSTER BY (category_id)""")
spark.sql("INSERT INTO workspace.default.day10_lc_items SELECT * FROM items")

cats = spark.createDataFrame([(i, f"cat_{i}") for i in range(50)], ["category_id", "category_name"])
cats.createOrReplaceTempView("cats")
plan = spark.sql("""EXPLAIN SELECT i.item_name, c.category_name
                    FROM workspace.default.day10_lc_items i JOIN cats c
                      ON i.category_id = c.category_id
                    WHERE i.category_id = 7""").collect()[0][0]
for kw in ["file pruning", "pruned", "Filter", "Scan"]:
    if kw in plan: print("plan mentions:", kw)
print("liquid-clustering table created; clustered files let the join skip non-matching files")

# --- Classic Hive/Spark bucketing (the interview answer for EMR / on-prem / Hive) ---------
# df_items.write.format("parquet")\
#     .bucketBy(8, "category_id").sortBy("category_id")\
#     .saveAsTable("bucketed_items")                      # pre-shuffles ONCE into 8 files/hash-buckets
# df_cats.write.format("parquet")\
#     .bucketBy(8, "category_id").sortBy("category_id")\
#     .saveAsTable("bucketed_cats")
# SELECT ... FROM bucketed_items i JOIN bucketed_cats c ON i.category_id = c.category_id
#   -> hash(category_id) % 8 sends matching rows of BOTH tables to the same bucket, so the join
#      reads bucket i of items + bucket i of cats and joins LOCALLY. No Exchange in the plan.

# INTERVIEW TALKING POINTS
# - Bucketing = one-time pre-shuffle traded for shuffle-free later joins. Verify it worked:
#   EXPLAIN shows no Exchange above the scans and the join directly on the bucketed scans.
# - Bucket count: a multiple of the executor core count (so each task reads ~1 bucket); too few
#   -> huge files/tasks; too many -> tiny files, metadata pressure. Re-bucketing after changing the
#   count rewrites everything (buckets are baked in) — a real operational cost to mention.
# - Skew trap: a dominant key hashes into ONE bucket -> that bucket's task straggles. Fix with
#   salting (Day 5/Day 8) or AQE skew-join handling (Day 8) — bucketing does NOT fix skew.
# - Databricks/UC today: Delta managed tables -> liquid clustering (CLUSTER BY), incremental;
#   ZORDER via OPTIMIZE is the older static layout (full rewrite, weighted interleave for
#   multiple columns). Ask first: 'is this Delta? then CLUSTER BY; Hive? then buckets.'
# - Bucketing also serves aggregation pruning (GROUP BY the bucket key) — mention for bonus points.

## Q8 — Tungsten & Whole-Stage Codegen `[ADVANCED]`

Your interviewer slides over a `SELECT category_id, COUNT(*) FROM items GROUP BY category_id` plan and asks: **"what physically happens, and what did Tungsten change?"**

1. Explain the Volcano iterator model and why it's slow (per-row virtual calls, no CPU cache locality).
2. Explain whole-stage codegen: what `*(n)` means in a physical plan, what gets fused, and the limits (some operators can't be codegen'd — e.g. format-specific readers).
3. Explain UnsafeRow / binary memory layout, on-heap vs off-heap (`OFF_HEAP`), and the vectorized cache.
4. How Photon changes this picture on Databricks.
5. Demonstrate checking all of this on a live plan (with the caveat that adaptive/Photon runtimes differ).

In [0]:
# Solution Q8 — explain + verify the compiled plan
from pyspark.sql import functions as F

items = spark.createDataFrame([(i, f"item_{i}", i % 50) for i in range(1000)],
                              ["item_id", "item_name", "category_id"])
items.createOrReplaceTempView("items")

plan = spark.sql("EXPLAIN FORMATTED SELECT category_id, COUNT(*) cnt FROM items GROUP BY category_id").collect()[0][0]
for kw in ["Photon", "WholeStageCodegen", "AdaptiveSparkPlan", "HashAggregate", "Exchange"]:
    if kw in plan: print("physical plan contains:", kw)
print("(On Photon-enabled Databricks runtimes, JVM codegen stages are replaced by vectorized C++ —\n"
      " the *concepts* below are what you explain in the interview.)")

# INTERVIEW TALKING POINTS — the 60-second Tungsten answer
# 1) VOLCANO MODEL: classic Spark evaluates a query as a chain of iterators — each operator calls
#    next() on the one below, PER ROW, through many virtual (hard-to-inline) calls. Row data hops
#    between heap objects -> pointer chasing, cache misses, GC pressure. At millions of rows/sec the
#    interpreter overhead dominates the actual work.
# 2) WHOLE-STAGE CODEGEN: Spark fuses a WHOLE stage of operators (scan -> filter -> aggregate
#    partial) into a single generated Java function (janino-compiled) that iterates the rows in a
#    tight loop with local variables — one call per BATCH of rows, values often staying in CPU
#    registers. In the classic plan, `*(2)` before an operator means "this operator is part of
#    generated codegen stage 2". Operators that can't be codegen'd (some file formats, UDFs) break
#    the fused chain and force the interpreter path — which is exactly why row-at-a-time Python
#    UDFs are slow (Q6).
# 3) UNSAFEROW / BINARY LAYOUT: Tungsten stores rows in contiguous off-heap-friendly byte arrays
#    — fixed-length fields in an 8-byte-aligned null bitset + values region, variable-length data
#    appended after — no Java objects, so no GC scanning and true cache locality. UnsafeExternalSort
#    and the shuffle both operate on these binary rows; OFF_HEAP memory (spark.memory.offHeap.enabled
#    + size) moves allocations out of the JVM heap entirely.
# 4) CACHE: df.cache() on Tungsten stores COMPRESSED COLUMNAR batches (highly compressed, vectorized
#    scans on read).
# 5) PHOTON (Databricks): replaces the JVM execution engine with a vectorized C++ engine — column
#    at a time, not row at a time — so on Databricks you often won't see WholeStageCodegen at all;
#    you'll see PhotonStage. Say: "Photon generalizes the Tungsten idea: columnar vectorized
#    execution instead of per-row codegen."
# 6) HOW TO VERIFY: EXPLAIN FORMATTED (above) — look for *(n) / PhotonStage markers; Spark UI SQL
#    tab shows the same DAG with codegen stage boxes.

## Q9 — Incremental Batch Ingestion with a Control-Table Ledger `[INTERMEDIATE]`

You ingest partner CSV/Parquet drops from a landing zone **without** Structured Streaming (the platform team wants plain batch jobs). Requirements:

1. Each hourly run picks up **only files never seen before** — no timestamps-in-filenames trust, no duplicate processing.
2. Re-running the job (crash, manual replay) must be a **no-op** for already-processed files.
3. Ops must be able to answer "when was file X loaded, by which batch, how many rows?"

Design the mechanism, implement it, and prove idempotency by running the same batch twice.

In [0]:
# Solution Q9 — control table as the source of truth for "what have I already seen?"
from pyspark.sql import functions as F
import os

SRC = "/Volumes/workspace/default/temp_parquet/day10/incremental_nb"
LEDGER = "workspace.default.day10_nb_ledger"
BRONZE = "workspace.default.day10_nb_bronze"
os.makedirs(SRC, exist_ok=True)

spark.sql(f"DROP TABLE IF EXISTS {LEDGER}")
spark.sql(f"DROP TABLE IF EXISTS {BRONZE}")
spark.sql(f"""CREATE TABLE {LEDGER}
              (file_path STRING NOT NULL, ingested_at TIMESTAMP, batch_id BIGINT, row_count BIGINT)""")
spark.sql(f"CREATE TABLE {BRONZE} (id BIGINT, val STRING, src_file STRING)")

def ingest_run(batch_id):
    """One scheduled run: anti-join the drop zone against the ledger, ingest only NEW files."""
    raw = spark.read.parquet(SRC).withColumn("_file", F.col("_metadata.file_path"))  # UC: no input_file_name()
    files = raw.select("_file").distinct().alias("f")
    ledger = spark.table(LEDGER).alias("l")
    new_files = [r["_file"] for r in files.join(ledger, files._file == ledger.file_path, "left_anti").collect()]
    if not new_files:
        print(f"batch {batch_id}: 0 new files -> NO-OP (idempotent re-run)")
        return 0
    new_df = raw.filter(F.col("_file").isin(new_files))
    n = new_df.count()
    (new_df.select("id", "val", F.col("_file").alias("src_file"))      # provenance kept in bronze
           .write.mode("append").saveAsTable(BRONZE))
    entries = (spark.createDataFrame([(f,) for f in new_files], "file_path string")
                   .withColumn("ingested_at", F.current_timestamp())
                   .withColumn("batch_id", F.lit(batch_id))
                   .withColumn("row_count", F.lit(int(n / len(new_files)))))
    entries.createOrReplaceTempView("entries")
    spark.sql(f"""MERGE INTO {LEDGER} l USING entries e ON l.file_path = e.file_path
                 WHEN NOT MATCHED THEN INSERT *""")            # crash between bronze write and this MERGE
                                                             # -> file re-processed; fix: same batch_id
                                                             #    + bronze PK MERGE instead of append
    print(f"batch {batch_id}: ingested {len(new_files)} new files, {n} rows")
    return len(new_files)

# --- simulate the partner dropping files ----------------------------------------------
(spark.range(5).withColumn("val", F.lit("a")).repartition(1).write.mode("append").parquet(SRC))
(spark.range(3).withColumn("val", F.lit("b")).repartition(1).write.mode("append").parquet(SRC))

n1 = ingest_run(1)            # first run: both files are new
n2 = ingest_run(2)            # RE-RUN with no new files: must be a no-op
assert n2 == 0

(spark.range(7).withColumn("val", F.lit("c")).repartition(1).write.mode("append").parquet(SRC))  # late file
n3 = ingest_run(3)            # third run: only the late file is new
assert n3 == 1

total = spark.sql(f"SELECT COUNT(*) c FROM {BRONZE}").collect()[0][0]
assert total == 15, total
spark.sql(f"SELECT batch_id, COUNT(*) files_ingested FROM {LEDGER} GROUP BY batch_id ORDER BY batch_id").show()
print(f"control-table ledger verified: bronze has exactly 15 rows after 3 runs + 1 duplicate run")

# INTERVIEW TALKING POINTS
# - Why not just Structured Streaming? Say: "Auto Loader + checkpoint does this natively; control
#   tables are the pattern when policy demands plain batch, or you also need cross-file state
#   (row counts, checksums, partner SLA reporting)." Both are file-LEDGER thinking.
# - Why not filename timestamps? Partners re-upload old files with new names, clocks skew, files
#   arrive late. The LEDGER (processed-file set) is the only durable truth.
# - Crash window: if we crash AFTER the bronze append but BEFORE the ledger MERGE, the next run
#   re-processes the file -> duplicate bronze rows. Production hardening: (a) MERGE into bronze on
#   a business key instead of append, or (b) write bronze + ledger in ONE foreachBatch-style
#   transaction, or (c) record batch_id and make bronze dedup idempotent. Mention this proactively —
#   it's the difference between "idempotent-ish" and idempotent.
# - Extensions ops will ask for: file row_count + checksum in the ledger (drift detection), a
#   `status` column (LOADED/QUARANTINED/FAILED) so the same table doubles as a DLQ ledger, and
#   backfill = point the ledger at a time range and DELETE rows to force re-processing.

## Q10 — The Double-Write Problem: Transactional Outbox `[ADVANCED]`

On every order event you must (1) **append to a Delta table** AND (2) **push to an external system** (a marketing API / S3 feed). The two systems cannot share a transaction, so a crash between them leaves them inconsistent — either the API missed events Delta has, or the API got duplicates.

Explain why two-phase commit / distributed transactions are a non-answer, then design an **exactly-once-effect** pipeline:

1. Which write is the transactional one, and what plays the role of the "outbox"?
2. How the external publisher derives its work FROM the outbox, and how it tracks what it already published.
3. What makes the external push itself idempotent (idempotency keys), and how replay/backfill works.

Demonstrate with Structured Streaming + `foreachBatch`.

In [0]:
# Solution Q10 — Delta append IS the transaction; external publish derives from it
from pyspark.sql import functions as F
import shutil

BASE = "/Volumes/workspace/default/temp_parquet/day10"
EVENTS = "workspace.default.day10_nb_outbox"
EXT_SINK = BASE + "/outbox_nb/external_sink"      # stands in for the external API/S3
for p in [BASE + "/outbox_nb/checkpoint", EXT_SINK]:
    try: shutil.rmtree(p, True)
    except Exception: pass
spark.sql(f"DROP TABLE IF EXISTS {EVENTS}")

rate = (spark.readStream.format("rate").option("rowsPerSecond", 5).load()
        .withColumn("payload", F.concat(F.lit("evt-"), F.col("value").cast("string"))))

def write_outbox(batch_df, batch_id):
    # ONE Delta append per micro-batch = the outbox record. The batch_id is the idempotency key.
    (batch_df.withColumn("batch_id", F.lit(batch_id))
            .write.format("delta").mode("append").saveAsTable(EVENTS))
    # Publish to the 'external system'. At-least-once is fine because the publisher can
    # reconstruct exactly what to send from the outbox table (see publisher phase below).
    (batch_df.select("payload").write.format("parquet").mode("append").save(EXT_SINK))

q = (rate.writeStream
     .foreachBatch(write_outbox)                       # foreachBatch: Delta txn per batch,
                                                      # exactly-once into the outbox
     .option("checkpointLocation", BASE + "/outbox_nb/checkpoint")
     .trigger(availableNow=True)                      # run-to-completion; prod = continuous
     .start())
q.awaitTermination()

n = spark.sql(f"SELECT COUNT(*) c FROM {EVENTS}").collect()[0][0]
print(f"outbox rows: {n}")

# --- PUBLISHER phase (what a separate job/daemon would do) -----------------------------
# SELECT payloads FROM outbox o LEFT ANTI JOIN publisher_ledger p ON o.batch_id = p.batch_id
# -> send to API with idempotency key (batch_id, value) -> MERGE batch_id into publisher_ledger.
# Crash anywhere: the next publisher pass re-reads the same outbox range; the API dedups on the
# idempotency key. NET effect: exactly-once delivery without any distributed transaction.

# INTERVIEW TALKING POINTS
# - Why not 2PC/XA: cross-system transaction coordinators add fragility + latency and APIs/S3
#   simply don't support them. The industry answer is the TRANSACTIONAL OUTBOX (from microservices
#   literature): make ONE local transactional write the source of truth, and derive everything
#   else from it asynchronously.
# - Mapping to streaming: Delta append per foreachBatch micro-batch (checkpoint ensures each batch
#   commits once) = the outbox. External publish is deliberately AT-LEAST-ONCE; the publisher
#   ledger + idempotency keys upgrade it to exactly-once EFFECT.
# - foreachBatch specifics to name: it runs on the driver per batch, gets a unique monotonically
#   increasing batch_id, and the SAME batch may be retried after failure — hence idempotent sinks
#   (MERGE on key / partitioned overwrite), never blind append.
# - Databricks-native alternatives worth naming: Delta + DLT expectations + Change Data Feed as the
#   outbox (publisher reads CDF, which gives you commit-versioned increments), or foreachBatch
#   with the Databricks SQL connector, or Kafka as the async spine with the same ledger logic.
# - Failure drill the interviewer will run: "publisher dies after sending 3 of 10 payloads" ->
#   restart re-sends all 10; API dedups 7; ledger records the batch done. Walk it out loud.

## Q11 — Schema Drift: `_rescued_data` Capture and Backfill `[ADVANCED]`

Your Auto Loader bronze pipeline runs with a **fixed, contracted schema** (`id`, `name`) — downstream consumers depend on it. Upstream silently starts emitting extra columns (`vip`, `bonus`) and occasionally type-changed fields. Requirements:

1. The pipeline must **not fail** and must **not silently drop** the new data.
2. New/drifted columns must be **captured with provenance** (which file they came from).
3. Once the schema contract is renegotiated, you must be able to **backfill** the captured data into properly typed columns.

Explain the difference between `_rescued_data` and the corrupt-record mechanism from Q5, and implement capture + backfill.

In [0]:
# Solution Q11 — Auto Loader with rescuedDataColumn, then a typed backfill pass
from pyspark.sql import functions as F, types as T
import shutil, os

BASE = "/Volumes/workspace/default/temp_parquet/day10"
os.makedirs(BASE + "/raw_json", exist_ok=True)
with open(BASE + "/raw_json/f1.json", "w") as f:
    f.write('{"id": 1, "name": "ann"}\n{"id": 2, "name": "bob"}\n')
with open(BASE + "/raw_json/f2.json", "w") as f:                 # upstream added vip/bonus
    f.write('{"id": 3, "name": "cat", "vip": true, "bonus": 9}\n{"id": 4, "name": "dan", "vip": false}\n')

for p in [BASE + "/al_nb/check", BASE + "/al_nb/schema"]:
    try: shutil.rmtree(p, True)
    except Exception: pass

fixed_schema = T.StructType([T.StructField("id", T.IntegerType()),
                             T.StructField("name", T.StringType())])
al = (spark.readStream.format("cloudFiles")
      .option("cloudFiles.format", "json")
      .schema(fixed_schema)                                   # THE contract: only id+name are typed
      .option("cloudFiles.rescuedDataColumn", "_rescued")    # anything else lands here, not dropped
      .option("cloudFiles.schemaLocation", BASE + "/al_nb/schema")
      .load(BASE + "/raw_json"))

q = (al.writeStream
     .foreachBatch(lambda df, bid: df.write.format("parquet").mode("append").save(BASE + "/al_nb/check/out"))
     .option("checkpointLocation", BASE + "/al_nb/check/chk")
     .trigger(availableNow=True)
     .start())
q.awaitTermination()

bronze = spark.read.parquet(BASE + "/al_nb/check/out")
bronze.show(truncate=False)                                   # rows 3,4 carry vip/bonus INSIDE _rescued
assert bronze.filter("_rescued IS NOT NULL").count() == 2
assert bronze.filter("_rescued IS NULL").count() == 2

# --- BACKFILL: contract renegotiated; promote rescued columns to typed columns ------------
rescued_schema = T.StructType([T.StructField("vip", T.BooleanType()),
                               T.StructField("bonus", T.LongType())])
backfilled = (bronze
    .withColumn("r", F.from_json(F.col("_rescued"), rescued_schema))
    .withColumn("vip",   F.coalesce(F.col("r.vip"),   F.lit(None).cast("boolean")))
    .withColumn("bonus", F.coalesce(F.col("r.bonus"), F.lit(None).cast("long")))
    .drop("r", "_rescued"))
backfilled.orderBy("id").show()
assert backfilled.filter("id = 3").collect()[0]["bonus"] == 9
assert backfilled.filter("id = 4").collect()[0]["vip"] is False
print("schema drift rescued + backfilled: contract intact, zero data lost")

# INTERVIEW TALKING POINTS
# - _rescued_data vs _corrupt_record (Q5): corrupt-record = the ROW could not be PARSED (truncated
#   JSON -> nulls + raw text). Rescued data = the row parsed fine, but a COLUMN didn't fit the
#   expected schema (unknown column, type mismatch) -> that column's value is parked in _rescued
#   as JSON, with _file_path provenance. Different failure modes, different mechanisms.
# - Why fixed schema in prod: contract stability for silver/gold + control over type inference
#   (cloudFiles.inferColumnTypes can silently retype). Rescue = escape hatch that neither fails
#   nor drops.
# - Operations: alert on `COUNT(_rescued IS NOT NULL) > threshold` — a non-empty rescue column IS
#   the schema-drift pager. Then: renegotiate contract -> evolve schema -> backfill from the
#   rescued JSON (this cell) or re-run Auto Loader over old files with a new schema.
# - Schema evolution knobs: cloudFiles.schemaEvolutionMode (addNewColumns / rescue / none) — with
#   rescue mode you get exactly this behavior; addNewColumns changes the schema automatically
#   (dangerous with fixed contracts).
# - Caveat: _rescued values are stringified; type fidelity survives only for what from_json can
#   re-parse — deep-nested type changes may need manual handling. Say that honestly.

## Q12 — Effective-Dated Currency Conversion in the Fact Pipeline `[INTERMEDIATE]`

Orders are booked in EUR; finance reports in USD. FX rates change intraday, and **the correct rate for a transaction is the latest rate effective at or before the transaction timestamp** — not the rate at batch time, not the current rate.

1. Design the `fx_rate` dimension (columns, the no-overlap invariant, how a new rate arrives).
2. Write the **range join** that converts each transaction correctly — and explain why an equi-join on `rate_date` is wrong (fan-out/precision bugs).
3. Show the MERGE that "closes" the previous rate row when a new rate arrives.
4. Add the data-quality check proving no overlapping rate windows exist.

In [0]:
# Solution Q12 — effective-dated rates + range join + MERGE to close rows
from pyspark.sql import functions as F
from datetime import datetime

FX = "workspace.default.day10_nb_fx"
spark.sql(f"""CREATE OR REPLACE TABLE {FX}
              (currency STRING, rate DOUBLE, effective_from TIMESTAMP, effective_to TIMESTAMP)""")
seed = [("EUR", 1.10, datetime(2026,1,1), datetime(2026,3,1)),
        ("EUR", 1.08, datetime(2026,3,1), None)]                    # NULL = still in effect
(spark.createDataFrame(seed, "currency string, rate double, effective_from timestamp, effective_to timestamp")
     .write.mode("append").saveAsTable(FX))

txs = spark.createDataFrame([
    (1, 100.0, datetime(2026,2,15)),   # falls in the 1.10 window
    (2, 200.0, datetime(2026,4,10)),  # falls in the 1.08 window
    (3,  50.0, datetime(2026,10,5)),  # must pick up the NEW rate below
], "tx_id int, amount_eur double, tx_ts timestamp")

# --- a new rate arrives: close the current row, then append (the SCD-2 "close + add" pattern) ---
spark.sql(f"""MERGE INTO {FX} t
             USING (SELECT 'EUR' AS currency, 1.02 AS rate,
                           timestamp'2026-10-01 00:00:00' AS effective_from) s
               ON t.currency = s.currency AND t.effective_to IS NULL
             WHEN MATCHED THEN UPDATE SET t.effective_to = s.effective_from""")
spark.sql(f"""INSERT INTO {FX}
             SELECT 'EUR', 1.02, timestamp'2026-10-01 00:00:00', NULL""")

# --- the range join: exactly ONE rate per transaction ----------------------------------
converted = (txs.alias("t")
    .join(spark.table(FX).alias("r"),
          (F.col("t.tx_ts") >= F.col("r.effective_from")) &
          (F.col("r.effective_to").isNull() | (F.col("t.tx_ts") < F.col("r.effective_to"))),
          "inner")
    .withColumn("amount_usd", F.round(F.col("amount_eur") * F.col("rate"), 2)))
converted.select("tx_id", "amount_eur", "rate", "amount_usd").show()

assert converted.count() == txs.count(), "fan-out bug! a tx joined >1 rate window"
usd = {r["tx_id"]: r["amount_usd"] for r in converted.collect()}
assert usd[1] == 110.0 and usd[2] == 216.0 and usd[3] == 51.0

# --- data-quality check: no overlapping windows per currency ----------------------------
overlaps = spark.sql(f"""SELECT a.effective_from, b.effective_from
                         FROM {FX} a JOIN {FX} b
                           ON a.currency = b.currency
                          AND a.effective_from < b.effective_from
                          AND a.effective_to > b.effective_from""")
assert overlaps.count() == 0
print("FX conversion verified: one rate per tx, no overlapping windows")

# INTERVIEW TALKING POINTS
# - Why not join on DATE(tx_ts) = rate_date: rates arrive intraday (a rate published 14:00 must NOT
#   apply to a 09:00 tx); midnight-aligned windows also break for late/adjusted feeds. The
#   effective_from/effective_to range join is timestamp-precise and standard for ALL effective-dated
#   dimensions (customer segments, pricing, tax rates).
# - The NULL effective_to trick (open interval) avoids inventing '9999-12-31' sentinels — but say
#   BOTH are accepted; sentinel is friendlier for some BI tools.
# - Fan-out is the #1 silent bug: if two windows overlap, a tx double-counts revenue. That's why the
#   DQ check (self-join for overlaps) belongs IN the pipeline, not in a yearly audit. Similar check
#   for gaps: every effective_from should equal the previous effective_to (lead/lag).
# - MERGE note: one source row can't both UPDATE the old row and INSERT the new one, so the close
#   and the insert are two statements (or a shuffle-union source). This trips people up live.
# - Performance: range joins are not equi-joins; on big tables use a coarse equi-pre-filter (currency,
#   or date bucket) + range condition, or Databricks range-join hints with a bounded interval.

## Q13 — Mini-Dimension for Fast-Changing Attributes `[ADVANCED]`

100M customers. Marketing wants to slice revenue by `age_band` × `income_band` × `city_tier` — but these demographic attributes change **monthly** for a meaningful share of customers. A plain SCD-2 customer dimension would explode to billions of rows (100M × monthly changes), bloating every join and every fact-row key.

1. Explain the Kimball **mini-dimension** pattern: what goes in it, how small it stays, and how the fact table references it.
2. Explain the **demographic outrigger** on the customer dimension and the **point-in-time bridge** — and why “revenue by current demographics” and “revenue by demographics at time of order” are DIFFERENT queries.
3. Implement a working miniature and run both analyses.

In [0]:
# Solution Q13 — mini-dim + fact carrying BOTH keys + point-in-time bridge
from pyspark.sql import functions as F

# 1) MINI-DIMENSION: one row per demographic PROFILE combination, surrogate-keyed.
#    Size = #combinations (4 here; realistically hundreds) — NOT #customers. It barely grows.
demo = spark.createDataFrame([
    (101, "18-25", "0-25k"), (102, "26-35", "25-50k"),
    (103, "36-45", "50-100k"), (104, "46-55", "100k+"),
], "demo_key int, age_band string, income_band string")

# 2) CUSTOMER DIM: SCD-1 on stable attrs + demographic OUTRIGGER (current profile pointer)
#    alice's outrigger points at her CURRENT profile (102) even though her older order (1001)
#    was placed under 101 — that difference is the whole point of this pattern.
cust = spark.createDataFrame([
    (1, "alice", 102),    # alice NOW 26-35/25-50k (moved from 101)
    (2, "bob",   102),
    (3, "carol", 104),
], "customer_key int, name string, current_demo_key int")

# 3) FACT carries BOTH keys: WHO (customer_key, stable) and WHAT DEMOGRAPHIC AT ORDER TIME (demo_key)
fact = spark.createDataFrame([
    (1001, 1, 101, 120.0, "2026-10-01"),   # alice ordered while 18-25/0-25k
    (1002, 2, 102,  80.0, "2026-10-02"),
    (1003, 3, 104,  60.0, "2026-10-03"),
    (1004, 1, 102,  40.0, "2026-10-20"),   # alice MOVED to 26-35/25-50k, then ordered again
], "order_key int, customer_key int, demo_key int, revenue double, order_date string")

# Analysis A — demographics AT TIME OF ORDER (from the fact's own demo_key): exact, historical
by_order_time = (fact.join(demo, "demo_key")
                      .groupBy("age_band")
                      .agg(F.round(F.sum("revenue"), 1).alias("revenue")))
by_order_time.show()
assert by_order_time.filter("age_band = '26-35'").collect()[0]["revenue"] == 120.0  # 80 + 40

# Analysis B — CURRENT demographics (fact -> customer outrigger -> mini-dim): re-attributes history
by_current = (fact.join(cust, "customer_key")
                  .join(demo, cust.current_demo_key == demo.demo_key)
                  .groupBy("age_band")
                  .agg(F.round(F.sum("revenue"), 1).alias("revenue")))
by_current.show()
assert by_current.filter("age_band = '18-25'").count() == 0  # alice no longer 18-25 -> history moved

# 4) POINT-IN-TIME BRIDGE (factless): what demographic did each customer hold WHEN
bridge = spark.createDataFrame([
    (1, 101, "2026-01-01", "2026-10-15"),   # alice was 18-25/0-25k until mid-Oct
    (1, 102, "2026-10-15", None),           # then moved
    (2, 102, "2026-01-01", None),
], "customer_key int, demo_key int, from_date string, to_date string")
print("bridge lets you answer 'demographics on date X' WITHOUT touching the fact table")

# INTERVIEW TALKING POINTS
# - Why not SCD-2: 100M customers × ~10% monthly demographic churn = ~10M new dim rows EVERY MONTH;
#   the dim triples in size yearly and every fact join gets slower. Mini-dim keeps the BIG dim
#   static (stable attrs) and the VOLATILE attrs in a tiny, bounded, cacheable table.
# - Fact grain decision: carrying BOTH customer_key and demo_key at order time is the Kimball
#   recommendation — the fact then self-documents history even if the bridge is lost.
# - The two analyses (A vs B) are the crux: "revenue by demographics at purchase" vs "revenue by
#   CURRENT demographics of buyers" answer different business questions — ALWAYS ask which one
#   product means before building.
# - Outrigger caveat: Kimball tolerates a mini-dim outrigger FROM the dimension, but chaining
#   dimension -> dimension -> dimension is an anti-pattern (snowflaking); keep it one hop.
# - Where it's used in the wild: telecom (plan changes), banking (risk bands), streaming
#   (subscription tier) — any hot-cycling segmentation attribute. Say one example from YOUR
#   projects (Tesla telemetry: vehicle config bands would be a good parallel).

## Q14 — Snowflake Table Types: Permanent vs Transient vs External (+ Cost) `[ADVANCED]`

You're migrating a lakehouse onto Snowflake and must decide, table by table, which type to use. For each of **permanent**, **transient**, and **external** tables explain:

1. What it is and how storage/Time Travel/Fail-Safe work differently for each.
2. The cost model implication (what do you pay for, and what can't you get back?).
3. A concrete placement decision for: bronze raw ingestion, a staging scratch table, and a legacy parquet archive in S3 that must stay in place.

Also: what are **hybrid tables** and when would you actually reach for one?

In [0]:
# Solution Q14 — explanatory (print-style: these are design/DDL answers, not runnable here)
answer = """
SNOWFLAKE TABLE TYPES — THE 60-SECOND MAP

1) PERMANENT (default)
   CREATE TABLE orders (...)  -- fully managed micro-partition storage
   - Time Travel: default 7 days, settable 0-90 via DATA_RETENTION_TIME_IN_DAYS
     -> query AS OF / AT(OFFSET) / BEFORE(statement) e.g.
        SELECT * FROM orders AT(TIMESTAMP => '2026-10-21 00:00:00');
   - Fail-Safe: 7 MORE days after retention ends, SNOWFLAKE-ONLY access (recovery
     from corruption/accidents). You cannot query it; you open a support case.
   - Cost: storage + BOTH windows hold dead micro-partitions -> 'clones, Time Travel
     and Fail-Safe all bill'. Dropped table still bills until retention + fail-safe expire.
   - Use: gold/semantic layers, anything with audit or recovery obligations.

2) TRANSIENT
   CREATE TRANSIENT TABLE staging_orders (...);
   - Time Travel: SUPPORTED but minimal (default 1 day, can lower to 0) --
     ALTER TABLE staging_orders SET DATA_RETENTION_TIME_IN_DAYS = 1;
   - NO Fail-Safe. That is the key line to say in the interview.
   - Cost: cheapest managed option; minimal retention billing; survives deletes/updates
     like a normal table.
   - Use: ETL scratch/staging, session temp data, anything reproducible from source.
     For pure within-session data, TEMPORARY tables (vanish with the session) are cheaper still.

3) EXTERNAL
   CREATE STAGE archive_stage URL='s3://legacy/orders/';
   CREATE EXTERNAL TABLE orders_archive (...) WITH LOCATION = @archive_stage
        FILE_FORMAT = (TYPE = PARQUET) [AUTO_REFRESH = TRUE];
   - Data STAYS in S3; Snowflake reads through the stage; no loading, no clones,
     no Time Travel, no Fail-Safe on history; metadata refreshes via event notifications
     (AUTO_REFRESH) or ALTER EXTERNAL TABLE ... REFRESH.
   - Cost: NO storage cost for data; you pay compute to query + the cloud provider's S3 cost.
   - Use: legacy archives that must not move, sharing curated parquet, cross-engine access.

4) HYBRID (Unistore)
   Row-level OLTP storage engine alongside columnar; supports primary keys /
   transactions for operational apps. Reach for it only when you genuinely run app
   workloads inside Snowflake; analytics tables stay columnar.

PLACEMENT FOR THE MIGRATION
  bronze raw ingestion  -> transient (rebuildable, no fail-safety needed, minimal cost)
  staging scratch       -> transient or TEMPORARY (session lifetime)
  legacy S3 parquet    -> external table (must stay in place, zero copy cost)
  gold/semantic        -> permanent (Time Travel + Fail-Safe + audit)

BONUS FACTS THAT WIN POINTS
  - Zero-copy clones copy ONLY metadata at creation; a clone of a transient table is transient.
  - DROP + UNDROP: within Time Travel you can UNDROP TABLE; after retention it's Fail-Safe/support.
  - Clones and streams interact: a stream on a table is not inherited by a clone.
  - Retention is a LEVER for cost control: dropping 90-day retention to 7 on huge tables
    is a classic, safe Snowflake cost cut.
"""
print(answer)

# INTERVIEW TALKING POINTS
# - Lead with the cost sentence: "Fail-Safe is permanent-table insurance; transient gives that up."
# - The classic follow-up: "why not make EVERYTHING transient?" -> no fail-safe = if storage
#   corrupts, Snowflake can't save you; audited/curated layers need the safety + travel.
# - Tie to Databricks for the comparison question: Delta tables on your own cloud storage =
#   you own Time Travel via VERSION AS OF + VACUUM retention; Snowflake manages storage for you.
#   Say one sentence on each — shows you know both stacks, not just one.

## Q15 — Snowflake Resource Monitors & Multi-Cluster Warehouses `[ADVANCED]`

Finance just discovered a single runaway query pattern burned **a month's credits in two days** on your shared `ANALYTICS_WH`. You own the fix. Explain and design:

1. **Resource monitors**: credit quotas, threshold triggers (`NOTIFY` / `SUSPEND` / `SUSPEND_IMMEDIATE`), the multi-threshold ladder (warn at 75%, suspend at 100%), and what exactly each trigger does to running queries.
2. **Multi-cluster warehouses**: scale-UP (bigger size) vs scale-OUT (`MIN_CLUSTER_COUNT`/`MAX_CLUSTER_COUNT`); what the `STANDARD` vs `ECONOMY` scaling policy does; why queues mean the size is wrong, not that you need more monitors.
3. Show the DDL that: creates a monitor scoped to the warehouse, alerts before suspension, and right-sizes the warehouse for a mixed concurrent workload.
4. The prevention story: query timeouts, per-role statement quotas, and how you'd attribute credits to teams afterwards.

In [0]:
# Solution Q15 — explanatory (print-style: governance design + DDL)
answer = """
1) RESOURCE MONITOR — the credit circuit breaker
   CREATE RESOURCE MONITOR analytics_wh_monitor WITH
     CREDIT_QUOTA = 5000                       -- monthly-ish budget for this warehouse
     TRIGGERS ON 75 PERCENT DO NOTIFY          -- email/integration warning
               ON 90 PERCENT DO SUSPEND        -- SUSPEND: stop NEW queries, let running ones finish
               ON 100 PERCENT DO SUSPEND_IMMEDIATE -- kill RUNNING queries mid-flight too;
               ON 120 PERCENT DO SUSPEND_IMMEDIATE;   -- hard ceiling (overage already burned)
   ALTER WAREHOUSE analytics_wh SET RESOURCE_MONITOR = analytics_wh_monitor;

   Semantics interviewers test:
   - SUSPEND: warehouse stops accepting new queries; queued + running queries FINISH first.
   - SUSPEND_IMMEDIATE: cancels running + queued queries NOW (that's the blast radius to admit).
   - A monitor resumes counting at the start of its billing window; credit quotas are warehouse-
     scoped (or account-scoped) -- a monitor can't protect a single query, it protects a BUDGET.
   - Monitor state (SUSPENDED) must be cleared manually (ALTER RESOURCE MONITOR ... RESUME);
     it does not auto-resume next window if the trigger was 100%.

2) MULTI-CLUSTER WAREHOUSES — throughput vs capacity
   ALTER WAREHOUSE analytics_wh SET
     WAREHOUSE_SIZE = 'XSMALL'        -- scale-UP unit: 2 credits/hr XS, doubling per size up
     MIN_CLUSTER_COUNT = 1            -- always-on floor (auto-suspend still saves idle credits)
     MAX_CLUSTER_COUNT = 4            -- scale-OUT ceiling: up to 4 concurrent clusters
     SCALING_POLICY = 'STANDARD';     -- STANDARD: spin up aggressively (latency-sensitive);
                                      -- ECONOMY: only when a cluster is FULLY utilized
                                      -- (cost-sensitive batch).
   - Scale-UP helps ONE big query (more servers per cluster); scale-OUT helps CONCURRENCY
     (many queries, each small). The runaway-query incident was scale: it starved everyone.
   - Queues with idle clusters = wrong sizing or MAX_CLUSTER too low; queues at MAX with
     high utilization = right time to scale out/up (check QUERY_QUEUE_TIME in QUERY_HISTORY).

3) PREVENTION (the answer finance actually wants)
   ALTER WAREHOUSE analytics_wh SET STATEMENT_TIMEOUT_IN_SECONDS = 3600;  -- kill runaways
   ALTER USER ... SET STATEMENT_QUEUED_TIMEOUT_IN_SECONDS = 900;          -- don't let users queue forever
   Per-role guards: give analysts a separate small warehouse with its own monitor instead of
   sharing the production WH (isolation IS the governance).

4) ATTRIBUTION — prove who burned what
   SELECT user_name, warehouse_name, SUM(credits_used_compute) credits
   FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ATtribution_BY_TAG  -- or WAREHOUSE_METERING_HISTORY
   ... GROUP BY 1, 2 ORDER BY credits DESC;
   Tag the warehouse, route teams to per-team warehouses, and the chargeback conversation
   turns into a dashboard instead of a meeting.
"""
print(answer)

# INTERVIEW TALKING POINTS
# - Structure the answer as: DETECT (monitors/ladder) -> CONTAIN (suspend semantics, blast radius)
#   -> PREVENT (timeouts, isolation) -> ATTRIBUTE (metering). Four beats, senior-sounding.
# - The classic trap: SUSPEND vs SUSPEND_IMMEDIATE semantics — 80% of candidates blur them.
# - Second trap: solving concurrency with a BIGGER warehouse (raises cost per SECOND) instead of
#   scale-out clusters. Say "queues are a sizing problem; measure QUERY_QUEUE_TIME first."
# - Cross-platform bonus: Databricks equivalent = job cluster isolation + serverless budgets /
#   tag-based chargeback with system.billing.usage (Day 9 cost-attribution question) — bridging
#   the two stacks is exactly what hybrid teams interview for.

## Q16 — Databricks Jobs: For-Each Tasks & Parameterized Fan-Out `[INTERMEDIATE]`

Your nightly pipeline processes **50 regions**, each needing the same notebook logic with a different `region_id` parameter. Today that's 50 manually cloned tasks — every new region means editing the job. Redesign it using Jobs' parameterization features:

1. **Parameterized notebook task** — how parameters flow in (widget/job parameters vs task values).
2. **For-each task** — loop over the output of a SQL/table task; what the `shared` config, `concurrency` limit, and task values give you; how a single region's failure is retried/repaired.
3. When you'd choose for-each vs **one job iterating internally** vs a **matrix** of parameters.
4. What changes for monitoring: per-iteration status, and how Repair Runs interact with the loop.

In [0]:
# Solution Q16 — explanatory (print-style: the job JSON you'd actually write)
answer = """
DESIGN: one SQL task lists the work -> one for-each task fans it out -> one dashboard sees it.

1) PARAMETERIZED NOTEBOOK TASK
   dbutils.widgets.get("region_id") inside the notebook; the job passes parameters per run:
     "parameters": {"region_id": "{{task.value.region_id}}"}
   Parameters are typed strings; job-level parameters set defaults; task-level values
   (set by an upstream task via dbutils.jobs.taskValues.set) override them per iteration.

2) FOR-EACH TASK (Jobs 2.x) — replaces 50 clones with ONE task definition
   "tasks": [
     {"task_key": "list_regions",
      "sql_task": {"warehouse_id": "...", "file": {"path": "list_regions.sql"}}},
        -- the SQL returns rows: region_id column; THAT RESULT SET is the loop domain
     {"task_key": "process_region",
      "for_each_task": {
        "inputs": "{{tasks.list_regions.values}}",     -- iterate the upstream rows
        "task": {
          "task_key": "region_inner",
          "notebook_task": {
            "notebook_path": "/etl/process_region",
            "parameters": {"region_id": "{{task.value.region_id}}"}},
          "timeout_seconds": 3600,
          "max_retries": 2 }},                        -- retries PER REGION, not per job
        "concurrency": 10,                             -- at most 10 regions in flight
        "shared": {"cluster": "..."}                   -- one cluster config shared by all
        }}]
   - The inner task body sees ONE value per iteration (task.value.region_id).
   - concurrency caps parallelism — protects the source DB / warehouse, prevents 50-way stampede.
   - A region that fails twice retried, then the for-each reports WHICH iteration failed;
     Repair Run lets you re-run ONLY the failed iteration(s), not all 50.

3) CHOOSING THE SHAPE
   - for-each: when the iteration list is DATA (rows in a table) and each unit is independent.
   - internal loop in one notebook: when units share expensive setup (one big cache), when you
     need custom scheduling inside, or iteration count is huge & trivial per unit (driver loop
     over 10k IDs beats a 10k-iteration task graph).
   - matrix: small FIXED cross-product (env x region) authored as a job matrix.
   - anti-pattern: 50 cloned tasks — say explicitly you'd delete that on day one.

4) MONITORING
   - Job run UI shows every inner iteration as its own cell with its own duration/retry state.
   - Emit per-iteration metrics via taskValues or a metrics table (Day 8 pattern) keyed by
     (run_id, region_id) — the fan-out is only useful if you can see per-region SLOs.
"""
print(answer)

# INTERVIEW TALKING POINTS
# - One-liner that lands: "the loop domain should be data, not job JSON — regions come from a
#   table, so onboarding region 51 is zero pipeline changes."
# - Concurrency is a real design decision: too high = source stampede + warehouse queueing;
#   too low = the nightly window misses. Anchor it to a source SLA, e.g. "we held 10 because the
#   source replica allows ~15 connections."
# - Pair with Run If dependencies (Day 6) and repair runs (Day 5) — failure handling composes:
#   retries per region -> repair the failed regions -> notify once. Mention that composition.
# - If asked about Airflow: the equivalent is dynamic task mapping (expand over a list), Day 18
#   below — say the pattern is identical across orchestrators; that's the insight they want.

## Q17 — Driver Logs, log4j & the 3 AM Evidence Trail `[ADVANCED]`

At 3 AM a streaming job dies: executor lost, query stopped, clusters re-scaled, and by morning the logs are **gone** because the cluster was terminated. Walk the full story:

1. **Where driver/executor logs live**: the UI, cluster log delivery, and what gets lost on termination.
2. **Python logging vs print**: why `print` from executors is useless, buffered, and unordered — and what a proper logging setup looks like (named loggers, levels, formatter, log4j bridge).
3. **What to log in a data pipeline**: the six things that make a 3 AM debugging session 5 minutes instead of 2 hours (batch ids, offsets, counts in/out, DLQ counts, durations, environment).
4. Set this up in code: structured per-batch logging + a metrics side-table so the evidence survives the cluster.

In [0]:
# Solution Q17 — logging setup + durable evidence table (runnable)
import logging, sys
from pyspark.sql import functions as F

# 1) PROPER LOGGER: named, formatted, stderr (shows in driver log output)
logger = logging.getLogger("nightly.pipeline")
if not logger.handlers:
    logger.setLevel(logging.INFO)
    h = logging.StreamHandler(sys.stderr)          # -> driver log stream, picked up by log4j bridge
    h.setFormatter(logging.Formatter("%(asctime)s | %(name)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)

# Try to quiet the noisy JVM side (serverless may manage this; guard it)
try:
    spark.sparkContext.setLogLevel("WARN")         # levels: ALL < DEBUG < INFO < WARN < ERROR < FATAL
    logger.info("spark log level set to WARN")
except Exception as e:
    logger.warning(f"could not set spark log level ({type(e).__name__}) — managed runtime")

# 2) WHAT TO LOG, per batch — the six fields that solve 3 AM pages
METRICS = "workspace.default.day10_nb_log_metrics"
spark.sql(f"CREATE TABLE IF NOT EXISTS {METRICS} (batch_id BIGINT, ts TIMESTAMP, rows_in BIGINT, rows_out BIGINT, dlq_rows BIGINT, duration_s DOUBLE, note STRING)")

def process_batch(batch_df, batch_id):
    import time
    t0 = time.time()
    n_in = batch_df.count()
    dlq = batch_df.filter(F.col("value") < 0)             # pretend DQ rule
    good = batch_df.subtract(dlq)
    # 3) dual-sink evidence: human-readable log NOW, queryable table FOREVER
    logger.info(f"batch {batch_id}: rows_in={n_in} dlq={dlq.count()} "
                f"duration={time.time()-t0:.2f}s")
    (spark.createDataFrame([(batch_id, None, n_in, good.count(), dlq.count(),
                             time.time()-t0, "nightly_run")],
                            "batch_id bigint, ts timestamp, rows_in bigint, rows_out bigint, dlq_rows bigint, duration_s double, note string")
        .withColumn("ts", F.current_timestamp())
        .write.mode("append").saveAsTable(METRICS))
    return n_in

demo = spark.createDataFrame([(10, 1), (-3, 2), (7, 3)], "value bigint, other bigint")   # -3 violates the DQ rule
assert process_batch(demo, 101) == 3
m = spark.sql(f"SELECT * FROM {METRICS} WHERE batch_id = 101").collect()[0]
assert (m["rows_in"], m["rows_out"], m["dlq_rows"]) == (3, 2, 1), m
spark.sql(f"SELECT * FROM {METRICS}").show(truncate=False)

# INTERVIEW TALKING POINTS — the 3 AM walkthrough
# - WHERE LOGS LIVE: Cluster UI -> Driver/Executor logs tabs (live cluster); **cluster log
#   delivery** (DBFS/UC volume/S3 destination configured at cluster creation) is the ONLY thing
#   that survives termination — say "we enable log delivery on all ephemeral clusters, so
#   termination never destroys the evidence." Serverless: driver std err surfaces in the
#   run output / job run UI.
# - PRINT vs LOGGING: print from EXECUTOR code buffers, interleaves unordered across JVM/Python
#   processes, and often never reaches the driver. logging with named loggers carries levels +
#   timestamps, routes to stderr (which the log4j bridge captures), and is greppable per logger.
# - LEVELS: log4j hierarchy (root -> child loggers); DEBUG in dev, WARN in prod; spark log level
#   via setLogLevel or log4j config; NEVER debug-level in prod (it can slow the job by itself).
# - THE SIX THINGS: (1) batch_id/micro-batch id, (2) source offsets / checkpoint version,
#   (3) rows in, (4) rows out, (5) DLQ/reject counts, (6) duration + environment (cluster id,
#   spark version). If you can compare rows_in to rows_out + dlq per batch, 90% of data incidents
#   become arithmetic.
# - Why the metrics TABLE alongside logs: logs are for the 3 AM human; the table is for the morning
#   post-mortem, the freshness/volume alerts (Day 6/8), and trend analysis. This dual-sink IS the
#   mature answer interviewers listen for.

## Q18 — Authoring an Airflow DAG for Databricks: TaskFlow, Deferrable Operators, SLAs `[ADVANCED]`

Your team runs Databricks pipelines but orchestrates cross-system workflows (Databricks → Snowflake → a vendor API → a report email) in **Airflow**. Whiteboard the DAG authoring choices:

1. **TaskFlow API** (`@task` decorators) vs classic operators — what XComs become, and the small-data-only rule for XComs.
2. **Deferrable operators & sensors** — what occupies an Airflow worker slot while a `DatabricksJobSensor` polls, how deferrable mode frees it (triggers running on the triggerer), and why that's the difference between a 100-DAG and a 10,000-DAG deployment.

3. **Failure semantics**: `retries` + `retry_delay` vs `execution_timeout` vs **SLA** (`sla` / `sla_miss_callback`) — three different clocks, three different audiences.

4. **Catchup & backfill**: what `catchup=True` does with `schedule`, and the safe way to backfill a month without drowning the warehouse.

5. When you'd still push orchestration DOWN into Databricks Jobs instead (hint: Day 6 comparison — but now from the authoring side).

In [0]:
# Solution Q18 — explanatory (print-style: Airflow is not installed here; this is the DAG you'd write)
answer = """
from airflow.decorators import dag, task
from airflow.providers.databricks.sensors.databricks_job import DatabricksJobSensor
from airflow.providers.databricks.operators.jobs import DatabricksRunNowOperator
from datetime import timedelta

@dag(schedule="@daily", catchup=False,                  # NEVER catchup=True blindly on a busy DAG
     default_args={"retries": 2, "retry_delay": timedelta(minutes=5),
                   "execution_timeout": timedelta(hours=2)},
     sla_miss_callback=notify_sla_channel, tags=["nightly"])
def nightly_cross_system():

    @task
    def extract_ids() -> list[str]:
        ...                                            # TaskFlow: return value auto-becomes
                                                       # an XCom; passing it to the next
                                                       # task resolves it. No xcom_pull boilerplate.
    @task
    def run_databricks(ids: list[str]):
        run = DatabricksRunNowOperator(
            task_id="dbx_pipeline", job_id=123,
            json_parameters={"entity_ids": ids}).execute({})  # parametrized Job (Q16!)
        return run.job_run_id

    @task.sensor(poke_interval=60, timeout=3600, mode="reschedule")
    def wait_for_dbx(job_run_id: str) -> bool:
        ...                                            # deferrable sensor: between polls the
                                                       # worker slot is RELEASED (triggerer polls)

    @task
def _(): pass

1) TASKFLOW vs classic: @task functions pass return values as implicit XComs — readable, typed,
   testable as plain functions. XCom RULE: values live in the metadata DB; keep them small
   (ids, paths, counts). Move DATA through systems (tables/files), never through Airflow.

2) DEFERRABLE: a classic sensor occupies a worker slot per poke. A deferrable one yields to
   the triggerer process (async) — one triggerer handles thousands of waits. At 10k DAGs this
   is the difference between a 3-worker and a 30-worker fleet. Same idea as smart sensors
   (deprecated) — say "deferrable operators are the modern answer" if asked.

3) THREE CLOCKS:
   - retries/retry_delay: task-level resilience (transient failures). Ask: is the failure
     transient? If not, retries just burn the window.
   - execution_timeout: hard stop on ONE task attempt (includes retries? NO — per try).
   - sla: BUSINESS clock measured from the DAG logical date; a miss fires sla_miss_callback
     but does NOT stop anything — it's a promise to stakeholders, not a failure.

4) CATCHUP/BACKFILL: with schedule + catchup=True Airflow creates every missed run since
   start_date — usually a stampede. Safe backfill: pause the DAG, run `airflow dags backfill
   -s ... -e ... --reset-dagruns` with limited parallelism, or better: parameterize the Job
   (Q16 pattern) and drive the date range INSIDE Databricks, so Airflow triggers ONE run.

5) PUSH-DOWN RULE (when Jobs beat Airflow): everything INSIDE the lakehouse (task DAGs with
   data dependencies, repair runs, for-each) belongs in a Databricks Job; Airflow stays the
   cross-system conductor (vendor APIs, Snowflake, emails, business calendars). One sentence:
   'Airflow decides WHEN across systems; Jobs decide HOW inside the lakehouse.'"""
print(answer)

# INTERVIEW TALKING POINTS
# - The three-clocks distinction (retries vs timeout vs SLA) is where candidates blur — slow down
#   there and give the audience for each: retries = the system, timeout = the task budget,
#   SLA = the business. Day 5 covered retries/repair in Jobs; today adds the Airflow side.
# - Deferrable operators: explain the MECHANISM (worker slot released, triggerer polls async) —
#   naming the triggerer process is the signal you've actually run it at scale.
# - XCom small-data rule — say it as a hard rule: "ids and paths, never rows."
# - Connect to Q16 explicitly: the Databricks job you trigger from Airflow should itself be the
#   for-each parameterized job — orchestration layers compose, they don't compete. That closing
#   sentence ("Airflow decides WHEN, Jobs decide HOW") is the one they'll remember.

---

## Day 10 Score Card

Attempt every question first, then grade yourself. Be honest — a 3/5 you mark as 5/5 stays broken forever.

| # | Question | Difficulty | Attempted | Solved | Time | Confidence (1-5) | Review topic? |
|---|---|---|---|---|---|---|---|
| Q1 | Conversion funnel (window functions) | INTERMEDIATE | ☐ | ☐ | ___ | ___ | ☐ |
| Q2 | Exact vs approx distinct count at scale | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q3 | Merge overlapping intervals | INTERMEDIATE | ☐ | ☐ | ___ | ___ | ☐ |
| Q4 | Running median (two heaps) | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q5 | Corrupt-record quarantine | INTERMEDIATE | ☐ | ☐ | ___ | ___ | ☐ |
| Q6 | Group-wise scoring with applyInPandas | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q7 | Bucketing vs liquid clustering | INTERMEDIATE | ☐ | ☐ | ___ | ___ | ☐ |
| Q8 | Tungsten & whole-stage codegen | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q9 | Ingestion control-table ledger | INTERMEDIATE | ☐ | ☐ | ___ | ___ | ☐ |
| Q10 | Transactional outbox pattern | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q11 | _rescued_data schema-drift backfill | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q12 | Effective-dated FX conversion | INTERMEDIATE | ☐ | ☐ | ___ | ___ | ☐ |
| Q13 | Mini-dimension for fast-changing attrs | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q14 | Snowflake table types | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q15 | Resource monitors & multi-cluster WH | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q16 | Jobs for-each tasks & fan-out | INTERMEDIATE | ☐ | ☐ | ___ | ___ | ☐ |
| Q17 | Driver logs & the evidence trail | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |
| Q18 | Airflow TaskFlow, deferrable, SLAs | ADVANCED | ☐ | ☐ | ___ | ___ | ☐ |

**Today's verdict** — INTERMEDIATE ≥ 6/7 and ADVANCED ≥ 8/11: on track for Week 3 (ADVANCED product-company level). Anything below: re-attempt that question tomorrow morning before Day 11.

**New concepts locked in today:** funnel vs cohort · sketch-based counting (HLL) · two-heap invariants · Arrow-batched pandas functions · shuffle-free joins · codegen/vectorized execution · file ledgers & outboxes · schema-drift rescue · effective-dated range joins · mini-dimensions · Snowflake cost governance · parameter fan-out · durable logging evidence · deferrable orchestration.